In [1]:
import numpy as np
if not hasattr(np, 'asfarray'):
    np.asfarray = lambda a, dtype=float: np.asarray(a, dtype=dtype)
print(f'NumPy {np.__version__} — asfarray shim applied for motmetrics compatibility.')

NumPy 2.4.6 — asfarray shim applied for motmetrics compatibility.


In [2]:
import json
from pathlib import Path
import pandas as pd
import motmetrics as mm


NATIVE_W, NATIVE_H = 1920, 1080  # MOT17 published annotation resolution — same for all sequences

ANALYTICS_DIR  = Path('../outputs/analytics')   # unused by Phase 5, kept for reference
VALIDATION_DIR = Path('../outputs/validation')
VALIDATION_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_DIR = VALIDATION_DIR
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SEQUENCES = [
    {
        'name'           : 'MOT17-04-FRCNN-fullres-bytetrack',
        'label'          : 'yolov8l + conf=0.25 + full-res + ByteTrack',
        'raw_tracks_json': OUTPUT_DIR / 'raw_tracks_MOT17-04-FRCNN-fullres.json',   # <-- was MOT17-02
        'gt_path'        : Path('../data/MOT17/train/MOT17-04-FRCNN/gt/gt.txt'),    # <-- was a .json
    },
    {
        'name'           : 'MOT17-02-FRCNN-fullres-bytetrack',
        'label'          : 'yolov8l + conf=0.25 + full-res + ByteTrack',
        'raw_tracks_json': OUTPUT_DIR / 'raw_tracks_MOT17-02-FRCNN-fullres.json',
        'gt_path'        : Path('../data/MOT17/train/MOT17-02-FRCNN/gt/gt.txt'),    # <-- was a .json
    },
    {
        'name'           : 'MOT17-02-FRCNN',
        'label'          : 'baseline (yolov8m, conf=0.40)',
        'raw_tracks_json': OUTPUT_DIR / 'raw_tracks_MOT17-02-FRCNN.json',
        'gt_path'        : Path('../data/MOT17/train/MOT17-02-FRCNN/gt/gt.txt'),
    },
    {
        'name'           : 'MOT17-02-FRCNN-conf025',
        'label'          : 'yolov8l + conf=0.25',
        'raw_tracks_json': OUTPUT_DIR / 'raw_tracks_MOT17-02-FRCNN-conf025.json',
        'gt_path'        : Path('../data/MOT17/train/MOT17-02-FRCNN/gt/gt.txt'),
    },
    {
        'name'           : 'MOT17-04-FRCNN',
        'label'          : 'baseline (yolov8m, conf=0.40)',
        'raw_tracks_json': OUTPUT_DIR / 'raw_tracks_MOT17-04-FRCNN.json',
        'gt_path'        : Path('../data/MOT17/train/MOT17-04-FRCNN/gt/gt.txt'),
    },
    {
        'name'           : 'MOT17-04-FRCNN-conf025',
        'label'          : 'yolov8l + conf=0.25',
        'raw_tracks_json': OUTPUT_DIR / 'raw_tracks_MOT17-04-FRCNN-conf025.json',
        'gt_path'        : Path('../data/MOT17/train/MOT17-04-FRCNN/gt/gt.txt'),
    },
]

print(f'Configured {len(SEQUENCES)} sequence(s):')
for seq in SEQUENCES:
    status = '✓' if seq['raw_tracks_json'].exists() else '✗ MISSING — run Phase 1 + archive first'
    print(f"  {seq['name']:<40} {status}")

Configured 6 sequence(s):
  MOT17-04-FRCNN-fullres-bytetrack         ✓
  MOT17-02-FRCNN-fullres-bytetrack         ✓
  MOT17-02-FRCNN                           ✓
  MOT17-02-FRCNN-conf025                   ✓
  MOT17-04-FRCNN                           ✓
  MOT17-04-FRCNN-conf025                   ✓


In [3]:
print(f'{"Sequence":<18}{"detect_w":>10}{"detect_h":>10}{"scale_x":>10}{"scale_y":>10}{"total_ids":>12}{"fps":>8}')
for seq in SEQUENCES:
    with seq['raw_tracks_json'].open() as f:
        d = json.load(f)
    meta = d['metadata']
    dw, dh = meta.get('frame_width'), meta.get('frame_height')
    sx, sy = NATIVE_W / dw, NATIVE_H / dh
    print(f"{seq['name']:<18}{dw:>10}{dh:>10}{sx:>10.2f}{sy:>10.2f}"
          f"{meta.get('total_unique_ids'):>12}{meta.get('fps'):>8.1f}")

Sequence            detect_w  detect_h   scale_x   scale_y   total_ids     fps
MOT17-04-FRCNN-fullres-bytetrack      1920      1080      1.00      1.00          72    30.0
MOT17-02-FRCNN-fullres-bytetrack      1920      1080      1.00      1.00         107    30.0
MOT17-02-FRCNN           960       540      2.00      2.00          51    30.0
MOT17-02-FRCNN-conf025       960       540      2.00      2.00          61    30.0
MOT17-04-FRCNN           960       540      2.00      2.00          59    30.0
MOT17-04-FRCNN-conf025       960       540      2.00      2.00          62    30.0


In [4]:
def evaluate_sequence(seq_config: dict) -> dict:
    name             = seq_config['name']
    raw_tracks_path  = seq_config['raw_tracks_json']
    gt_path          = seq_config['gt_path']

    assert raw_tracks_path.exists(), (
        f"Missing: {raw_tracks_path}\n"
        f"Run Phase 1 on {name}'s video, then run the archive cell for it first."
    )
    assert gt_path.exists(), f'Missing ground truth: {gt_path}'

    with raw_tracks_path.open('r', encoding='utf-8') as f:
        tracks_data = json.load(f)

    # --- scale hypothesis boxes to native GT resolution ---
    detect_w = tracks_data['metadata'].get('frame_width')
    detect_h = tracks_data['metadata'].get('frame_height')
    assert detect_w and detect_h, (
        f'{raw_tracks_path} is missing frame_width/frame_height metadata — '
        f're-run Phase 1 with the updated TrackStore.'
    )
    scale_x = NATIVE_W / detect_w
    scale_y = NATIVE_H / detect_h

    rows = []
    for tid, records in tracks_data['tracks'].items():
        for rec in records:
            frame = rec['frame'] + 1  # gt.txt is 1-indexed
            x1, y1, x2, y2 = rec['bbox']
            x1, y1, x2, y2 = x1 * scale_x, y1 * scale_y, x2 * scale_x, y2 * scale_y
            w, h = x2 - x1, y2 - y1
            rows.append([frame, int(tid), x1, y1, w, h, 1, -1, -1, -1])

    hyp_df = pd.DataFrame(rows, columns=[
        'frame', 'id', 'bb_left', 'bb_top', 'bb_width', 'bb_height', 'conf', 'x', 'y', 'z'
    ]).sort_values(['frame', 'id'])

    hyp_path = OUTPUT_DIR / f'{name}_hypothesis.txt'
    hyp_df.to_csv(hyp_path, header=False, index=False)

    # --- filter GT to pedestrians only ---
    gt_raw = pd.read_csv(gt_path, header=None,
                          names=['frame', 'id', 'bb_left', 'bb_top', 'bb_width', 'bb_height',
                                 'conf', 'class', 'visibility'])
    gt_pedestrians = gt_raw[(gt_raw['class'] == 1) & (gt_raw['conf'] == 1)].copy()
    gt_filtered_path = OUTPUT_DIR / f'{name}_gt_pedestrians_filtered.txt'
    gt_pedestrians.to_csv(gt_filtered_path, header=False, index=False)

    # --- frame alignment sanity check ---
    gt_mm  = mm.io.loadtxt(str(gt_filtered_path), fmt='mot16')
    hyp_mm = mm.io.loadtxt(str(hyp_path), fmt='mot16')

    gt_frames  = set(gt_mm.index.get_level_values('FrameId'))
    hyp_frames = set(hyp_mm.index.get_level_values('FrameId'))
    overlap    = gt_frames & hyp_frames
    assert len(overlap) > 0.8 * len(gt_frames), (
        f'[{name}] Frame overlap suspiciously low ({len(overlap)}/{len(gt_frames)}) — '
        f'check the +1 frame-index offset.'
    )

    # --- box-size sanity check — catches the exact 2x scale bug class before
    #     trusting the MOTA number, instead of discovering it after the fact ---
    gt_w_avg  = gt_pedestrians['bb_width'].mean()
    hyp_w_avg = hyp_df['bb_width'].mean()
    size_ratio = gt_w_avg / hyp_w_avg
    if not (0.7 <= size_ratio <= 1.3):
        print(f"  ⚠️  [{name}] GT/Hyp width ratio is {size_ratio:.2f} (expected ~1.0) "
              f"— results below may still be scale-mismatched, inspect before trusting.")

    # --- run motmetrics ---
    acc = mm.utils.compare_to_groundtruth(gt_mm, hyp_mm, 'iou', distth=0.5)
    mh  = mm.metrics.create()
    summary = mh.compute(acc, metrics=mm.metrics.motchallenge_metrics, name=name)

    result = {
        'name'                : name,
        'label' : seq_config.get('label', ''),
        'mota'                : summary['mota'].iloc[0] * 100,
        'motp'                : summary['motp'].iloc[0] * 100 if not np.isnan(summary['motp'].iloc[0]) else float('nan'),
        'idf1'                : summary['idf1'].iloc[0] * 100,
        'num_switches'        : int(summary['num_switches'].iloc[0]),
        'recall'              : summary['recall'].iloc[0] * 100,
        'precision'           : summary['precision'].iloc[0] * 100,
        'mostly_tracked'      : int(summary['mostly_tracked'].iloc[0]),
        'partially_tracked'   : int(summary['partially_tracked'].iloc[0]),
        'mostly_lost'         : int(summary['mostly_lost'].iloc[0]),
        'num_frames_evaluated': len(overlap),
        'size_ratio_gt_hyp'   : round(size_ratio, 3),
    }

    print(f"[{name}] ({seq_config.get('label','')})  MOTA={result['mota']:.2f}%  MOTP={result['motp']:.3f}  "
      f"IDF1={result['idf1']:.2f}%  IDs={result['num_switches']}  "
      f"Recall={result['recall']:.1f}%  Precision={result['precision']:.1f}%")
    return result

print('evaluate_sequence() defined.')

evaluate_sequence() defined.


In [5]:
results = []
for seq in SEQUENCES:
    print(f"\n=== Evaluating {seq['name']} ===")
    res = evaluate_sequence(seq)
    results.append(res)

results_df = pd.DataFrame(results)
results_df


=== Evaluating MOT17-04-FRCNN-fullres-bytetrack ===
[MOT17-04-FRCNN-fullres-bytetrack] (yolov8l + conf=0.25 + full-res + ByteTrack)  MOTA=30.46%  MOTP=14.313  IDF1=44.90%  IDs=20  Recall=32.5%  Precision=94.2%

=== Evaluating MOT17-02-FRCNN-fullres-bytetrack ===
  ⚠️  [MOT17-02-FRCNN-fullres-bytetrack] GT/Hyp width ratio is 0.59 (expected ~1.0) — results below may still be scale-mismatched, inspect before trusting.
[MOT17-02-FRCNN-fullres-bytetrack] (yolov8l + conf=0.25 + full-res + ByteTrack)  MOTA=21.86%  MOTP=17.056  IDF1=34.59%  IDs=48  Recall=28.1%  Precision=82.5%

=== Evaluating MOT17-02-FRCNN ===
  ⚠️  [MOT17-02-FRCNN] GT/Hyp width ratio is 0.60 (expected ~1.0) — results below may still be scale-mismatched, inspect before trusting.
[MOT17-02-FRCNN] (baseline (yolov8m, conf=0.40))  MOTA=16.34%  MOTP=20.884  IDF1=26.82%  IDs=44  Recall=26.8%  Precision=72.4%

=== Evaluating MOT17-02-FRCNN-conf025 ===
  ⚠️  [MOT17-02-FRCNN-conf025] GT/Hyp width ratio is 0.69 (expected ~1.0) — res

,name,label,mota,motp,idf1,num_switches,recall,precision,mostly_tracked,partially_tracked,mostly_lost,num_frames_evaluated,size_ratio_gt_hyp
0,MOT17-04-FRCNN-fullres-bytetrack,yolov8l + conf=0.25 + full-res + ByteTrack,30.458187,14.313430,44.897449,20,32.504153,94.192919,10,28,45,1050,0.832
1,MOT17-02-FRCNN-fullres-bytetrack,yolov8l + conf=0.25 + full-res + ByteTrack,21.861041,17.055559,34.585258,48,28.050159,82.546722,6,19,37,600,0.587
2,MOT17-02-FRCNN,"baseline (yolov8m, conf=0.40)",16.344653,20.883848,26.821608,44,26.833863,72.355246,8,15,39,598,0.600
3,MOT17-02-FRCNN-conf025,yolov8l + conf=0.25,13.599914,21.302434,28.327361,53,31.284646,64.260447,7,21,34,598,0.685
4,MOT17-04-FRCNN,"baseline (yolov8m, conf=0.40)",27.667851,17.980502,39.641247,42,32.220283,87.831021,7,33,43,1048,0.849
5,MOT17-04-FRCNN-conf025,yolov8l + conf=0.25,29.324810,17.640403,40.947552,45,33.547112,89.043925,9,29,45,1048,0.854


In [6]:
avg_mota    = results_df['mota'].mean()
avg_motp    = results_df['motp'].mean()
avg_idf1    = results_df['idf1'].mean()
total_idsw  = int(results_df['num_switches'].sum())
DASH = '\u2014'

print('=== Results by Configuration ===')
for label in results_df['label'].unique():
    subset = results_df[results_df['label'] == label]
    print(f"\n[{label}]")
    print(f"  Sequences : {', '.join(subset['name'])}")
    print(f"  Avg MOTA  : {subset['mota'].mean():.2f}%")
    print(f"  Avg Recall: {subset['recall'].mean():.1f}%")
    print(f"  Avg Prec. : {subset['precision'].mean():.1f}%")
    print(f"  Total IDs : {int(subset['num_switches'].sum())}")

print()
print('=== Baseline vs. Experiment, Per Sequence ===')
for base_name in ['MOT17-02-FRCNN', 'MOT17-04-FRCNN']:
    base = results_df[results_df['name'] == base_name].iloc[0]
    exp  = results_df[results_df['name'] == f'{base_name}-conf025'].iloc[0]
    delta = exp['mota'] - base['mota']
    print(f"{base_name}: {base['mota']:.2f}% -> {exp['mota']:.2f}%  (Δ {delta:+.2f})")

print()
print('=== Comparison Against Published Baseline ===')
print(f'{"Metric":<20}{"Ours (avg)":>15}{"Zidani et al. 2024":>25}')
print(f'{"":<20}{"":>15}{"(MOT17 avg, all seq.)":>25}')
print('-' * 60)
print(f'{"MOTA":<20}{avg_mota:>14.2f}%{"63.82%":>25}')
print(f'{"HOTA":<20}{"N/A (not computed)":>15}{"43.15":>25}')
print(f'{"IDF1":<20}{avg_idf1:>14.2f}%{DASH:>25}')
print(f'{"Total ID Switches":<20}{total_idsw:>15}{DASH:>25}')

print()
print(f'Caveat: this average is over {len(results_df)} of MOT17\'s 7 training sequences')
print('(the 2 static-camera ones, matching our own fixed-CCTV deployment). Zidani et al.')
print('likely averaged across all 7 (or all 21 detector-variant combinations). Still a')
print('directional comparison, not a byte-exact reproduction of their methodology.')

summary_path = OUTPUT_DIR / 'mot17_multi_sequence_summary.csv'
results_df.to_csv(summary_path, index=False)
print(f'\nSaved: {summary_path}')

=== Results by Configuration ===

[yolov8l + conf=0.25 + full-res + ByteTrack]
  Sequences : MOT17-04-FRCNN-fullres-bytetrack, MOT17-02-FRCNN-fullres-bytetrack
  Avg MOTA  : 26.16%
  Avg Recall: 30.3%
  Avg Prec. : 88.4%
  Total IDs : 68

[baseline (yolov8m, conf=0.40)]
  Sequences : MOT17-02-FRCNN, MOT17-04-FRCNN
  Avg MOTA  : 22.01%
  Avg Recall: 29.5%
  Avg Prec. : 80.1%
  Total IDs : 86

[yolov8l + conf=0.25]
  Sequences : MOT17-02-FRCNN-conf025, MOT17-04-FRCNN-conf025
  Avg MOTA  : 21.46%
  Avg Recall: 32.4%
  Avg Prec. : 76.7%
  Total IDs : 98

=== Baseline vs. Experiment, Per Sequence ===
MOT17-02-FRCNN: 16.34% -> 13.60%  (Δ -2.74)
MOT17-04-FRCNN: 27.67% -> 29.32%  (Δ +1.66)

=== Comparison Against Published Baseline ===
Metric                   Ours (avg)       Zidani et al. 2024
                                       (MOT17 avg, all seq.)
------------------------------------------------------------
MOTA                         23.21%                   63.82%
HOTA              